# Q-SAW final experiments: quantum stochastic walks for attack-graph risk ranking
One notebook, run once with **Save Version -> Save & Run All (Commit)**.
Settings: Accelerator **GPU (T4 / P100)**, **Internet ON**, inputs attached:
the XSB archive dataset (`XSB-5.1.0-source.bin`) and the improved CIC-IDS2017/CSE-CIC-IDS2018 dataset.

Part 1  MulVAL multi-stage attack graphs (main benchmark): 4 sizes x 10 fresh graphs,
        3 attacker models (budget 10, 30, unlimited), 3 robustness tests, decoherence analysis.
Part 2  Flow datasets (QAGC's data family), strictly unsupervised protocol v4.1.
Every parameter is fixed in CFG below before the run; ground truth is used only for scoring.

In [1]:
# 0. configuration and environment
import os, re, gc, glob, time, json, heapq, shutil, zipfile, tarfile, pickle, zlib, warnings, ipaddress
import tracemalloc, subprocess
from collections import deque
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, scipy.sparse as sp, scipy.linalg as sla
from scipy import stats
warnings.filterwarnings("ignore")

CFG = dict(
    OUT=os.environ.get("QSAW_OUT", "/kaggle/working/qsaw_final"),
    RUN_MULVAL=os.environ.get("QSAW_RUN_MULVAL", "1") == "1",
    RUN_FLOW=os.environ.get("QSAW_RUN_FLOW", "1") == "1",
    # ---- Part 1 (protocol fixed before the run; seeds 0-1 and 100-107 were development only)
    SIZES=[25, 50, 75, 100], N_GRAPHS=10, SEED0=200, MAX_TRIES=24, MULVAL_TIMEOUT=2400,
    GT_SAMPLES=20000, GT_SEED=12345, BUDGETS=[10, 30, None],          # None = unlimited attacker
    OMEGAS=[round(0.1 * i, 1) for i in range(11)], MAIN_OMEGA=0.5, DT=0.05,
    NOISE=[0.25, 0.5], MISSING=0.2, MC_BASE=[100, 1000],
    # ---- Part 2 (flow datasets)
    FLOW_ROOT=os.environ.get("QSAW_FLOW_ROOT",
                             "/kaggle/input/datasets/ernie55ernie/improved-cicids2017-and-csecicids2018"),
    FLOW_DATASETS=["CIC2017", "CSE2018"], FLOW_SIZES={50: "5min", 200: "5min", 500: "15min"},
    FLOW_MAXWIN=int(os.environ.get("QSAW_FLOW_MAXWIN", 20)),
)
for k in ("SIZES", "N_GRAPHS", "SEED0", "GT_SAMPLES"):          # test overrides only
    if os.environ.get(f"QSAW_{k}"):
        CFG[k] = json.loads(os.environ[f"QSAW_{k}"])
OUT = CFG["OUT"]; os.makedirs(OUT, exist_ok=True)
TAG = f"{zlib.crc32(json.dumps({k: CFG[k] for k in CFG if k not in ('OUT', 'RUN_MULVAL', 'RUN_FLOW')}, sort_keys=True, default=str).encode()):08x}"
CACHE = f"{OUT}/cache_{TAG}"; FIG = f"{OUT}/figures"
os.makedirs(CACHE, exist_ok=True); os.makedirs(FIG, exist_ok=True)
REPORT = f"{OUT}/RESULTS.md"; open(REPORT, "w").write("# Q-SAW results\n\n")
DT = CFG["DT"]; NEG = -1e300
NCPU = os.cpu_count() or 1


def say(*a):
    s = " ".join(str(x) for x in a); print(s)
    with open(REPORT, "a") as f:
        f.write(s + "\n")


def table(df, title=None, digits=4):
    if title: say(f"\n### {title}")
    say("```\n" + df.round(digits).to_string() + "\n```")


def save(obj, path):
    pickle.dump(obj, open(path + ".tmp", "wb")); os.replace(path + ".tmp", path)


try:
    import torch
    GPU = torch.cuda.is_available()
except Exception:
    torch = None; GPU = False
USE_TORCH = (torch is not None) and (GPU or os.environ.get("QSAW_FORCE_TORCH") == "1")
DEV = "cuda" if GPU else "cpu"
mem_gb = int(open("/proc/meminfo").read().split()[1]) / 2**20 if os.path.exists("/proc/meminfo") else float("nan")
say(f"[env] GPU: {torch.cuda.get_device_name(0) if GPU else 'NONE'} | quantum batch backend: "
    f"{'torch-' + DEV if USE_TORCH else 'numpy (slow)'} | CPU cores: {NCPU} | RAM: {mem_gb:.0f} GB | cache: {CACHE}")
if not GPU:
    say("[warn] No GPU: the quantum walks will run on CPU and the notebook may take many hours. "
        "Settings -> Accelerator -> GPU.")


[env] GPU: Tesla T4 | quantum batch backend: torch-cuda | CPU cores: 4 | RAM: 31 GB | cache: /kaggle/working/qsaw_final/cache_f46fe741


In [2]:
# 1. quantum stochastic walk engine (exact Lindblad, O(N^2) memory) + self-tests
def build_operators(n, src, dst, wgt):
    """H = normalised symmetric adjacency (coherent part); G[dst,src] = w/s_out(src) (jump rates); d = exit rates"""
    W = sp.csr_matrix((np.asarray(wgt, float), (np.asarray(src), np.asarray(dst))), shape=(n, n))
    W.setdiag(0.0); W.eliminate_zeros()
    s = np.asarray(W.sum(axis=1)).ravel(); inv = np.zeros(n); inv[s > 0] = 1 / s[s > 0]
    G = (sp.diags(inv) @ W).T.tocsr(); d = np.asarray(G.sum(axis=0)).ravel()
    A = (W + W.T).tocsr(); deg = np.asarray(A.sum(axis=1)).ravel()
    di = np.zeros(n); di[deg > 0] = 1 / np.sqrt(deg[deg > 0])
    return (sp.diags(di) @ A @ sp.diags(di)).tocsr(), G, d


def _rhs(rho, H, G, d, w):
    out = np.zeros_like(rho)
    if w < 1:
        Hr = H @ rho; out += (-1j * (1 - w)) * (Hr - Hr.conj().T)
    if w > 0:
        p = rho.real.diagonal()
        out -= (0.5 * w) * (d[:, None] * rho + rho * d[None, :])
        out[np.diag_indices_from(out)] += w * (G @ p)
    return out


def qsw_numpy(H, G, d, rho0, w, T):
    """RK4, complex128; returns the time-averaged (Cesaro) populations over [0, T]"""
    rho = rho0.astype(complex).copy(); acc = np.zeros(rho.shape[0])
    for _ in range(int(round(T / DT))):
        k1 = _rhs(rho, H, G, d, w); s2 = rho + 0.5 * DT * k1
        k2 = _rhs(s2, H, G, d, w); s3 = rho + 0.5 * DT * k2
        k3 = _rhs(s3, H, G, d, w); s4 = rho + DT * k3
        k4 = _rhs(s4, H, G, d, w)
        acc += (DT / 6) * (rho.real.diagonal() + 2 * s2.real.diagonal() + 2 * s3.real.diagonal() + s4.real.diagonal())
        rho = rho + (DT / 6) * (k1 + 2 * k2 + 2 * k3 + k4); rho = 0.5 * (rho + rho.conj().T)
    return acc / T


def _tsparse(M):
    c = M.tocoo()
    idx = torch.tensor(np.vstack([c.row, c.col]).astype(np.int64))
    return torch.sparse_coo_tensor(idx, torch.tensor(c.data.astype(np.float32)), c.shape).coalesce().to(DEV)


def qsw_batch(H, G, d, rho0, omegas, T):
    """all decoherence rates at once; torch complex64 on GPU, numpy fallback otherwise -> (len(omegas), n)"""
    if not USE_TORCH:
        return np.stack([qsw_numpy(H, G, d, rho0, w, T) for w in omegas])
    n = H.shape[0]; B = len(omegas); c64 = torch.complex64
    Hs, Gs = _tsparse(H), _tsparse(G)
    dd = torch.tensor(d, dtype=torch.float32, device=DEV); w = torch.tensor(omegas, dtype=torch.float32, device=DEV)
    rho = torch.tensor(rho0, dtype=c64, device=DEV).unsqueeze(0).repeat(B, 1, 1)
    coh = (-1j * (1 - w)).to(c64)[:, None, None]; dis = (0.5 * w)[:, None, None]

    def hmul(r):
        X = torch.cat([r.real, r.imag], 0).permute(1, 0, 2).reshape(n, -1)
        Y = torch.sparse.mm(Hs, X).reshape(n, 2 * B, n).permute(1, 0, 2)
        return torch.complex(Y[:B].contiguous(), Y[B:].contiguous())

    def rhs(r):
        Hr = hmul(r)
        o = coh * (Hr - Hr.mH) - dis * (dd[None, :, None] * r + r * dd[None, None, :])
        p = r.diagonal(dim1=-2, dim2=-1).real
        o.diagonal(dim1=-2, dim2=-1).add_((w[:, None] * torch.sparse.mm(Gs, p.T.contiguous()).T).to(c64))
        return o

    dg = lambda r: r.diagonal(dim1=-2, dim2=-1).real
    acc = torch.zeros(B, n, device=DEV)
    for _ in range(int(round(T / DT))):
        k1 = rhs(rho); s2 = rho + 0.5 * DT * k1
        k2 = rhs(s2); s3 = rho + 0.5 * DT * k2
        k3 = rhs(s3); s4 = rho + DT * k3
        k4 = rhs(s4)
        acc += (DT / 6) * (dg(rho) + 2 * dg(s2) + 2 * dg(s3) + dg(s4))
        rho = rho + (DT / 6) * (k1 + 2 * k2 + 2 * k3 + k4); rho = 0.5 * (rho + rho.mH)
    out = (acc / T).cpu().numpy().astype(float)
    del rho, k1, k2, k3, k4, s2, s3, s4
    if GPU: torch.cuda.empty_cache()
    return out


def engine_self_test():
    rng = np.random.default_rng(0); worst = dict(rhs=0.0, rk4=0.0, markov=0.0, batch=0.0)
    for _ in range(6):
        n = int(rng.integers(4, 9)); m = 3 * n; s = rng.integers(0, n, m); t = rng.integers(0, n, m); k = s != t
        s, t, x = s[k], t[k], rng.uniform(0.1, 1.0, k.sum())
        H, G, d = build_operators(n, s, t, x); Hd, Gd = H.toarray(), G.toarray()
        rho0 = np.zeros((n, n), complex); rho0[0, 0] = 0.6; rho0[1, 1] = 0.4
        for w in (0.0, 0.5, 1.0):
            I = np.eye(n); L = -1j * (1 - w) * (np.kron(I, Hd) - np.kron(Hd.T, I))
            for i in range(n):
                for j in range(n):
                    if Gd[i, j] > 0:
                        Lk = np.zeros((n, n)); Lk[i, j] = np.sqrt(Gd[i, j]); LdL = Lk.T @ Lk
                        L += w * (np.kron(Lk, Lk) - 0.5 * np.kron(I, LdL) - 0.5 * np.kron(LdL.T, I))
            X = rng.normal(size=(n, n)) + 1j * rng.normal(size=(n, n)); R = X @ X.conj().T; R /= np.trace(R)
            worst["rhs"] = max(worst["rhs"], np.abs(_rhs(R, H, G, d, w) - (L @ R.reshape(-1, order="F")).reshape(n, n, order="F")).max())
            ts = np.linspace(0, 3.0, 3001); E = sla.expm(L * (ts[1] - ts[0])); v = rho0.reshape(-1, order="F"); ps = []
            for _t in ts:
                ps.append(np.real(np.diag(v.reshape(n, n, order="F")))); v = E @ v
            exact = np.trapezoid(np.array(ps), ts, axis=0) / 3.0
            worst["rk4"] = max(worst["rk4"], np.abs(qsw_numpy(H, G, d, rho0, w, 3.0) - exact).max())
        p = sla.expm((Gd - np.diag(d)) * 0.01)
        cls = np.zeros(n); pt = np.real(np.diag(rho0)).copy()
        for _t in range(300):
            cls += 0.01 * 0.5 * pt; pt = p @ pt; cls += 0.01 * 0.5 * pt
        worst["markov"] = max(worst["markov"], np.abs(qsw_numpy(H, G, d, rho0, 1.0, 3.0) - cls / 3.0).max())
        if USE_TORCH:
            bt = qsw_batch(H, G, d, rho0, [0.0, 0.5, 1.0], 3.0)
            for i, w in enumerate((0.0, 0.5, 1.0)):
                worst["batch"] = max(worst["batch"], np.abs(bt[i] - qsw_numpy(H, G, d, rho0, w, 3.0)).max())
    say(f"[self-test] O(N^2) Lindblad RHS vs full N^2xN^2 superoperator: {worst['rhs']:.1e} (must be < 1e-10)")
    say(f"[self-test] RK4 Cesaro populations vs exact expm:             {worst['rk4']:.1e} (must be < 1e-4)")
    say(f"[self-test] omega=1 vs classical master equation:              {worst['markov']:.1e} (must be < 1e-3)")
    if USE_TORCH:
        say(f"[self-test] torch complex64 batch vs numpy complex128:          {worst['batch']:.1e} (must be < 1e-4)")
    assert worst["rhs"] < 1e-10 and worst["rk4"] < 1e-4 and worst["markov"] < 1e-3 and worst["batch"] < 1e-4
    say("[self-test] PASSED")


engine_self_test()


[self-test] O(N^2) Lindblad RHS vs full N^2xN^2 superoperator: 8.3e-17 (must be < 1e-10)
[self-test] RK4 Cesaro populations vs exact expm:             5.5e-08 (must be < 1e-4)
[self-test] omega=1 vs classical master equation:              2.4e-06 (must be < 1e-3)
[self-test] torch complex64 batch vs numpy complex128:          1.5e-07 (must be < 1e-4)
[self-test] PASSED


In [3]:
# 2. MulVAL: build (idempotent), network generator, run, parse
def sh(cmd, timeout=None):
    return subprocess.run(["bash", "-c", cmd], capture_output=True, text=True, timeout=timeout)


W_ = "/kaggle/working"


def ensure_mulval():
    if os.path.isfile(f"{W_}/mulval/bin/attack_graph") and glob.glob(f"{W_}/XSB/bin/xsb"):
        return "already built"
    if not glob.glob(f"{W_}/XSB/build/configure"):
        src = glob.glob("/kaggle/input/**/XSB*", recursive=True)
        if not src:
            raise SystemExit("XSB archive not found: attach the dataset containing XSB-5.1.0-source.bin")
        shutil.rmtree(f"{W_}/xsbx", ignore_errors=True); os.makedirs(f"{W_}/xsbx")
        (zipfile.ZipFile(src[0]) if zipfile.is_zipfile(src[0]) else tarfile.open(src[0])).extractall(f"{W_}/xsbx")
        cfg = glob.glob(f"{W_}/xsbx/**/build/configure", recursive=True)
        shutil.rmtree(f"{W_}/XSB", ignore_errors=True); shutil.move(os.path.dirname(os.path.dirname(cfg[0])), f"{W_}/XSB")
    script = r'''
    cd /kaggle/working
    apt-get -qq update > /dev/null 2>&1; apt-get -qq install -y build-essential flex bison default-jdk graphviz > /dev/null 2>&1
    export JAVA_HOME=$(dirname $(dirname $(readlink -f $(which javac)))); cp $JAVA_HOME/include/linux/jni_md.h $JAVA_HOME/include/ 2>/dev/null
    [ -d mulval ] || git clone -q https://github.com/risksense/mulval.git
    if [ -z "$(find XSB -path '*bin/xsb' 2>/dev/null)" ]; then chmod -R u+x XSB/build; (cd XSB/build && CFLAGS="-fcommon -O2" ./configure > /tmp/xc.log 2>&1 && ./makexsb > /tmp/xm.log 2>&1) || { tail -20 /tmp/xm.log; exit 1; }; fi
    export PATH=$(dirname $(find /kaggle/working/XSB -path "*bin/xsb" | head -1)):$PATH
    cd mulval/src/attack_graph
    sed -i 's/^\tg++ -g -DLINUX/\tg++ -std=gnu++11 -fpermissive -fcommon -Wl,--allow-multiple-definition -g -DLINUX/' Makefile
    sed -i 's/^\tgcc -g -c lex.yy.c/\tgcc -fcommon -g -c lex.yy.c/' Makefile
    sed -i 's/^\tmv graphit.tab.h y.tab.cpp.h/\tcp graphit.tab.h y.tab.cpp.h/' Makefile
    sed -i 's/fa >0;/fa != 0;/g' attack_graph.cpp
    cd /kaggle/working; mkdir -p mulval/bin/adapter mulval/bin/metrics; [ -d mulval/bin/attack_graph ] && rmdir mulval/bin/attack_graph
    export MULVALROOT=/kaggle/working/mulval; (cd mulval && make > /tmp/mm.log 2>&1) || { grep -i -A3 " error" /tmp/mm.log | tail -20; exit 1; }
    '''
    r = sh(script)
    if r.returncode != 0:
        raise SystemExit("MulVAL build failed:\n" + r.stdout[-2000:] + r.stderr[-1000:])
    return "built"


def rule_probs(path):
    txt = re.sub(r"/\*.*?\*/", "", open(path).read(), flags=re.S)
    txt = "\n".join(l for l in txt.splitlines() if not l.strip().startswith("%"))
    out = []
    for p in txt.split("interaction_rule(")[1:]:
        m = re.search(r"rule_desc\(\s*'(.*?)'\s*,\s*([0-9.]+)\s*\)", p, flags=re.S)
        out.append((m.group(1), float(m.group(2))) if m else ("", 1.0))
    return out


KNOBS = dict(p_vuln=0.30, p_local=0.30, p_client=0.40, p_incompetent=0.30, p_reuse=0.20, p_admin_cache=0.15, p_leak=0.02)
CVSS_W = dict(AV={"N": 0.85, "L": 0.55}, AC={"L": 0.77, "H": 0.44}, PR={"N": 0.85, "L": 0.62}, UI={"N": 0.85, "R": 0.62})


def exploit_prob(rng, av, ui):
    """CVSS v3.1 exploitability 8.22*AV*AC*PR*UI normalised by its maximum 3.887; AC, PR sampled from CVSS categories"""
    ac = rng.choice(["L", "H"], p=[0.8, 0.2]); pr = rng.choice(["N", "L"], p=[0.6, 0.4])
    return round(min(1.0, 8.22 * CVSS_W["AV"][av] * CVSS_W["AC"][ac] * CVSS_W["PR"][pr] * CVSS_W["UI"][ui] / 3.887), 4)


def gen_network(H, seed, k=KNOBS):
    rng = np.random.default_rng(seed)
    nd = max(2, round(0.06 * H)); ns = max(2, round(0.12 * H)); nw = max(2, H - nd - ns - 2)
    dmz = [f"{['web', 'mail', 'vpn'][i % 3]}{i}" for i in range(nd)]
    srv = [f"{['db', 'file', 'app'][i % 3]}{i}" for i in range(ns)]
    ndep = max(1, round(nw / 15)); ws = [f"ws_d{i % ndep}_{i}" for i in range(nw)]
    dept = {w: int(w.split("_")[1][1:]) for w in ws}; admin = ["dc0", "adminws0"]; hosts = dmz + srv + ws + admin
    zone = {**{h: "dmz" for h in dmz}, **{h: "srv" for h in srv}, **{h: "lan" for h in ws}, **{h: "adm" for h in admin}}
    F, P, svc = ["attackerLocated(internet).", "hacl(H,H,_,_)."], {}, []
    F += [f"attackGoal(execCode({h},_))." for h in hosts]

    def add_svc(h, prog, port, perm):
        svc.append((h, prog, port, perm)); F.append(f"networkServiceInfo({h},{prog},tcp,{port},{perm}).")

    for h in dmz:
        t = h.rstrip("0123456789")
        if t == "web": add_svc(h, "httpd", 80, "user"); F.append(f"isWebServer({h}).")
        if t == "mail": add_svc(h, "smtpd", 25, "user")
        if t == "vpn": add_svc(h, "vpnService", 443, "root")
    for h in srv:
        t = h.rstrip("0123456789")
        add_svc(h, {"db": "mysqld", "file": "smbd", "app": "tomcat"}[t], {"db": 3306, "file": 445, "app": 8080}[t],
                "root" if t == "file" else "user")
        add_svc(h, "sshd", 22, "root")
    for h in ws:
        add_svc(h, "smbd", 445, "root")
        if rng.random() < 0.3: add_svc(h, "sshd", 22, "user")
    add_svc("dc0", "ldapd", 389, "root"); add_svc("dc0", "sshd", 22, "root"); add_svc("adminws0", "sshd", 22, "root")
    for (h, prog, port, perm) in svc:
        if rng.random() < k["p_vuln"]:
            v = f"v_{h}_{prog}"; P[v] = exploit_prob(rng, "N", "N")
            F += [f"vulExists({h},{v},{prog}).", f"vulProperty({v},remoteExploit,privEscalation)."]
    for h in hosts:
        if rng.random() < k["p_local"]:
            v = f"l_{h}"; P[v] = exploit_prob(rng, "L", "N")
            F += [f"vulExists({h},{v},kernel).", f"vulProperty({v},localExploit,privEscalation)."]
    for h in ws:
        u = f"u_{h}"; F.append(f"hasAccount({u},{h},user).")
        if rng.random() < k["p_incompetent"]: F.append(f"inCompetent({u}).")
        if rng.random() < k["p_client"]:
            v = f"c_{h}"; P[v] = exploit_prob(rng, "N", "R")
            F += [f"vulExists({h},{v},browser).", f"vulProperty({v},remoteClient,privEscalation)."]
        same = [x for x in ws if dept[x] == dept[h] and x != h]
        if same and rng.random() < k["p_reuse"]: F.append(f"hasAccount({u},{same[rng.integers(len(same))]},user).")
        if rng.random() < k["p_admin_cache"]: F.append(f"hasAccount(itadmin,{h},root).")
    F += [f"hasAccount(itadmin,{h},root)." for h in srv + admin]
    A = set()

    def allow(a, b, port): A.add((a, b, "tcp", port))

    for h in dmz:
        allow("internet", h, {"web": 80, "mail": 25, "vpn": 443}[h.rstrip("0123456789")])
    for h in srv:
        t = h.rstrip("0123456789")
        if t == "db":
            for x in dmz + srv:
                if x.startswith("web") or x.startswith("app"): allow(x, h, 3306)
        if t == "app":
            for x in dmz:
                if x.startswith("web"): allow(x, h, 8080)
            for x in ws: allow(x, h, 8080)
        if t == "file":
            for x in ws: allow(x, h, 445)
        allow(h, "dc0", 389); allow("adminws0", h, 22)
    for x in dmz:
        if x.startswith("vpn"):
            for w in ws: allow(x, w, 445); allow(x, w, 22)
    for w in ws:
        allow(w, "dc0", 389); allow("adminws0", w, 22); allow("adminws0", w, 445)
        for y in ws:
            if y != w and dept[y] == dept[w]: allow(w, y, 445); allow(w, y, 22)
    allow("adminws0", "dc0", 22); allow("adminws0", "dc0", 389)
    ports = {"dmz": [80, 25, 443, 22], "srv": [3306, 445, 8080, 22], "lan": [445, 22], "adm": [22, 389]}
    for a in hosts:
        for b in hosts:
            if a != b and zone[a] != zone[b] and rng.random() < k["p_leak"] / max(1, len(hosts) / 50):
                allow(a, b, int(rng.choice(ports[zone[b]])))
    F += [f"hacl({a},{b},{p},{q})." for (a, b, p, q) in sorted(A)]
    for w in ws:
        F.append(f"hacl({w},internet,httpProtocol,httpPort).")
        for x in dmz:
            if x.startswith("web"): F.append(f"hacl({w},{x},httpProtocol,httpPort).")
    return "\n".join(F) + "\n", dict(hosts=hosts, zone=zone, vuln_p=P)


def run_mulval(facts, workdir, timeout):
    shutil.rmtree(workdir, ignore_errors=True); os.makedirs(workdir)
    open(f"{workdir}/input.P", "w").write(facts)
    t0 = time.time()
    subprocess.run(["bash", "-c", f"cd {workdir} && graph_gen.sh input.P -v > run.log 2>&1"],
                   env=ENV, capture_output=True, text=True, timeout=timeout)
    el = time.time() - t0
    if not os.path.exists(f"{workdir}/VERTICES.CSV"):
        raise RuntimeError("no graph: " + open(f"{workdir}/run.log").read()[-300:].replace("\n", " "))
    V = pd.read_csv(f"{workdir}/VERTICES.CSV", header=None, names=["id", "label", "type", "p0"])
    E = pd.read_csv(f"{workdir}/ARCS.CSV", header=None, names=["parent", "child", "w"])
    return V, E, el


def build_graph(V, E, meta):
    """MulVAL arcs point from a node to its preconditions; attack flow = child -> parent"""
    ids = V["id"].to_numpy(); ix = {v: i for i, v in enumerate(ids)}
    lab = V["label"].tolist(); typ = np.array(V["type"].tolist()); n = len(V)
    par = E["parent"].map(ix).to_numpy(); chi = E["child"].map(ix).to_numpy()
    and_p = np.ones(n); leaf_p = np.ones(n); vuln = np.zeros(n, bool)
    for i in range(n):
        if typ[i] == "AND":
            m = re.match(r"RULE (\d+)", lab[i]); k = int(m.group(1)) if m else -1
            and_p[i] = RULES[k][1] if 0 <= k < len(RULES) else 1.0
        if typ[i] == "LEAF" and lab[i].startswith("vulExists("):
            vuln[i] = True; leaf_p[i] = meta["vuln_p"].get(lab[i][len("vulExists("):-1].split(",")[1].strip("'"), 1.0)
    host_nodes = {}
    for i in range(n):
        m = re.match(r"execCode\(([^,]+),", lab[i])
        if m and typ[i] == "OR": host_nodes.setdefault(m.group(1), []).append(i)
    start = [i for i in range(n) if lab[i].startswith("attackerLocated(")]
    return dict(n=n, label=lab, type=typ, par=par, chi=chi, and_p=and_p, leaf_p=leaf_p, vuln=vuln,
                host_nodes=host_nodes, start=start, hosts=meta["hosts"], zone=meta["zone"])


def depth_stats(g):
    adj = [[] for _ in range(g["n"])]
    for p, c in zip(g["par"], g["chi"]): adj[c].append(p)
    d = np.full(g["n"], -1); q = deque(g["start"])
    for s in g["start"]: d[s] = 0
    while q:
        u = q.popleft()
        for v in adj[u]:
            if d[v] < 0: d[v] = d[u] + 1; q.append(v)
    st = [min(d[i] for i in idx) // 2 for idx in g["host_nodes"].values() if min(d[i] for i in idx) >= 0]
    return (int(np.median(st)), int(max(st))) if st else (0, 0)


In [4]:
# 3. ground truth: Monte-Carlo attackers over MulVAL AND/OR semantics
def mc_attacker(g, S, seed, budget=None, chunk=2000, probs=None):
    """budget=None: every enabled exploit is attempted once (least fixpoint, = unlimited attacker).
    budget=B: the attacker makes B attempts, each time picking uniformly among currently enabled,
    not-yet-tried rules. Exploit outcomes ~ Bernoulli(rule probability x CVSS exploit probability).
    Returns {host: P(code execution on host)}."""
    and_p, leaf_p = probs if probs is not None else (g["and_p"], g["leaf_p"])
    n, typ = g["n"], g["type"]; rng = np.random.default_rng(seed)
    C = sp.csr_matrix((np.ones(len(g["par"]), np.float32), (g["chi"], g["par"])), shape=(n, n))
    nchild = np.bincount(g["par"], minlength=n)
    A = np.flatnonzero(typ == "AND"); O = np.flatnonzero(typ == "OR"); L = np.flatnonzero(typ == "LEAF")
    hosts = list(g["host_nodes"]); hits = np.zeros(len(hosts)); done = 0
    cnt_of = lambda X: np.asarray(X.astype(np.float32) @ C)
    while done < S:
        s = min(chunk, S - done)
        X = np.zeros((s, n), bool); X[:, L] = rng.random((s, len(L))) < leaf_p[L]
        fire = rng.random((s, len(A))) < and_p[A]
        if budget is None:
            for _ in range(2 * n + 2):
                cnt = cnt_of(X)
                newA = (cnt[:, A] == nchild[A]) & fire & ~X[:, A]; newO = (cnt[:, O] > 0) & ~X[:, O]
                if not newA.any() and not newO.any(): break
                X[:, A] |= newA; X[:, O] |= newO
        else:
            tried = np.zeros((s, len(A)), bool)
            for _ in range(n):
                newO = (cnt_of(X)[:, O] > 0) & ~X[:, O]
                if not newO.any(): break
                X[:, O] |= newO
            for _ in range(budget):
                avail = (cnt_of(X)[:, A] == nchild[A]) & ~tried & ~X[:, A]
                r = np.flatnonzero(avail.any(1))
                if len(r) == 0: break
                ch = np.where(avail[r], rng.random((len(r), len(A))), -1.0).argmax(1)
                tried[r, ch] = True; ok = fire[r, ch]
                X[r[ok], A[ch[ok]]] = True
                for _ in range(n):
                    newO = (cnt_of(X)[:, O] > 0) & ~X[:, O]
                    if not newO.any(): break
                    X[:, O] |= newO
        for j, h in enumerate(hosts):
            hits[j] += X[:, g["host_nodes"][h]].any(1).sum()
        done += s
    return {h: hits[j] / S for j, h in enumerate(hosts)}


In [5]:
# 4. ranking methods on attack graphs (identical inputs: graph + step probabilities + attacker start)
def walk_graph(g):
    """attack-flow graph for walks/search: child -> parent, LEAF nodes dropped except the attacker start;
    an edge into an AND step carries its success probability (rule p x CVSS p of its vulnerability leaves)"""
    typ, n = g["type"], g["n"]; q = g["and_p"].copy()
    for p, c in zip(g["par"], g["chi"]):
        if typ[p] == "AND" and typ[c] == "LEAF": q[p] *= g["leaf_p"][c]
    keep = np.flatnonzero((typ != "LEAF") | np.isin(np.arange(n), g["start"]))
    new = -np.ones(n, int); new[keep] = np.arange(len(keep))
    m = (new[g["par"]] >= 0) & (new[g["chi"]] >= 0)
    par, chi = g["par"][m], g["chi"][m]
    w = np.where(typ[par] == "AND", q[par], 1.0)
    return dict(n=len(keep), src=new[chi], dst=new[par], w=w.astype(float), start=[int(new[s]) for s in g["start"]],
                q=np.full(len(g["start"]), 1 / len(g["start"])),
                hosts={h: [int(new[i]) for i in idx if new[i] >= 0] for h, idx in g["host_nodes"].items()})


def host_scores(node_score, wg):
    return {h: float(max(node_score[i] for i in idx)) if idx else NEG for h, idx in wg["hosts"].items()}


def t_rule_dir(wg):
    """label-free time scale: mean directed hop distance from the attacker over reachable nodes"""
    adj = [[] for _ in range(wg["n"])]
    for a, b in zip(wg["src"], wg["dst"]): adj[a].append(b)
    dist = np.full(wg["n"], -1); q = deque(wg["start"])
    for s in wg["start"]: dist[s] = 0
    while q:
        u = q.popleft()
        for v in adj[u]:
            if dist[v] < 0: dist[v] = dist[u] + 1; q.append(v)
    r = dist[dist > 0]
    return max(DT, round(max(float(r.mean()) if len(r) else 1.0, 1.0) / DT) * DT)


def order_scores(wg, dfs, undirected=False):
    adj = [set() for _ in range(wg["n"])]
    for a, b in zip(wg["src"], wg["dst"]):
        adj[a].add(b)
        if undirected: adj[b].add(a)
    adj = [sorted(x) for x in adj]
    starts = [wg["start"][i] for i in np.argsort(-np.asarray(wg["q"]), kind="stable")]
    sc = np.full(wg["n"], NEG); seen = np.zeros(wg["n"], bool); k = 0
    if dfs:
        for s0 in starts:
            st = [s0]
            while st:
                u = st.pop()
                if seen[u]: continue
                seen[u] = True; sc[u] = -k; k += 1
                st += [v for v in reversed(adj[u]) if not seen[v]]
    else:
        dq = deque()
        for s0 in starts: seen[s0] = True; dq.append(s0)
        while dq:
            u = dq.popleft(); sc[u] = -k; k += 1
            for v in adj[u]:
                if not seen[v]: seen[v] = True; dq.append(v)
    return sc


def astar_scores(wg, undirected=False, normalise=False):
    """A* with no goal set (h = 0): most probable path from the start(s); score = -best path cost.
    cost = -log(w) (attack graphs) or -log(w / strength) (flow graphs); start cost -log q"""
    nb = [dict() for _ in range(wg["n"])]
    for a, b, x in zip(wg["src"], wg["dst"], wg["w"]):
        nb[a][b] = nb[a].get(b, 0) + x
        if undirected: nb[b][a] = nb[b].get(a, 0) + x
    strength = np.array([sum(d.values()) for d in nb])
    dist = np.full(wg["n"], np.inf); h = []
    for s0, q0 in zip(wg["start"], wg["q"]):
        c0 = -np.log(max(q0, 1e-300)); dist[s0] = min(dist[s0], c0); heapq.heappush(h, (dist[s0], s0))
    while h:
        c, u = heapq.heappop(h)
        if c > dist[u]: continue
        for v, x in nb[u].items():
            nc = c - np.log(max(x / strength[u] if normalise else x, 1e-300))
            if nc < dist[v] - 1e-15: dist[v] = nc; heapq.heappush(h, (nc, v))
    return np.where(np.isfinite(dist), -dist, NEG)


def aostar_scores(g):
    """exact minimal-cost AND/OR solution graph for every node (Knuth's generalised Dijkstra = all-nodes AO*):
    OR = min over children, AND = -log p_rule + sum of children, vulnerability leaf = -log p_cvss"""
    n, typ = g["n"], g["type"]
    kids = [[] for _ in range(n)]; parents = [[] for _ in range(n)]
    for p, c in zip(g["par"], g["chi"]): kids[p].append(c); parents[c].append(p)
    cost = np.full(n, np.inf); done = np.zeros(n, bool); rem = np.array([len(k) for k in kids]); acc = np.zeros(n); h = []
    for i in range(n):
        if typ[i] == "LEAF": cost[i] = -np.log(max(g["leaf_p"][i], 1e-300)); heapq.heappush(h, (cost[i], i))
    while h:
        c, u = heapq.heappop(h)
        if done[u] or c > cost[u]: continue
        done[u] = True
        for p in parents[u]:
            if done[p]: continue
            if typ[p] == "AND":
                rem[p] -= 1; acc[p] += c
                if rem[p] == 0:
                    cost[p] = acc[p] - np.log(max(g["and_p"][p], 1e-300)); heapq.heappush(h, (cost[p], p))
            elif c < cost[p]:
                cost[p] = c; heapq.heappush(h, (c, p))
    sc = np.where(np.isfinite(cost), -cost, NEG)
    return {hh: float(max(sc[i] for i in idx)) for hh, idx in g["host_nodes"].items()}


def pagerank_scores(wg, alpha=0.85):
    n = wg["n"]; Wm = sp.csr_matrix((wg["w"], (wg["src"], wg["dst"])), shape=(n, n))
    s = np.asarray(Wm.sum(axis=1)).ravel(); inv = np.zeros(n); inv[s > 0] = 1 / s[s > 0]
    PT = (sp.diags(inv) @ Wm).T.tocsr(); v = np.zeros(n); v[wg["start"]] = wg["q"]; x = v.copy(); dang = s == 0
    for _ in range(1000):
        xn = alpha * (PT @ x) + (alpha * x[dang].sum() + 1 - alpha) * v
        if np.abs(xn - x).sum() < 1e-12: x = xn; break
        x = xn
    return x


def assetrank_style_scores(g, d=0.85):
    """dependency PageRank in the spirit of AssetRank (Sawilla & Ou, 2008), implemented from its description:
    rank flows from attacker goals to preconditions; AND passes full rank to each precondition, OR splits it"""
    n, typ = g["n"], g["type"]; outdeg = np.bincount(g["par"], minlength=n)
    wt = np.where(typ[g["par"]] == "AND", 1.0, 1.0 / np.maximum(outdeg[g["par"]], 1))
    M = sp.csr_matrix((wt, (g["chi"], g["par"])), shape=(n, n))
    e = np.zeros(n); goals = [i for idx in g["host_nodes"].values() for i in idx]; e[goals] = 1 / max(len(goals), 1)
    x = e.copy()
    for _ in range(500):
        xn = d * (M @ x) + (1 - d) * e; xn /= xn.sum()
        if np.abs(xn - x).sum() < 1e-12: x = xn; break
        x = xn
    return {h: float(max(x[i] for i in idx)) for h, idx in g["host_nodes"].items()}


def qagc_struct(wg):
    """arc space + spectrum shared by the QAGC walks (paper: weighted Grover coin, flip-flop shift, H = gamma*A)"""
    n = wg["n"]
    A_ = sp.csr_matrix((np.r_[wg["w"], wg["w"]], (np.r_[wg["src"], wg["dst"]], np.r_[wg["dst"], wg["src"]])), shape=(n, n))
    A_.sum_duplicates(); Ad = A_.toarray(); A_ = A_.tocoo(); au, av, aw = A_.row, A_.col, A_.data
    s = np.bincount(au, weights=aw, minlength=n); q = np.sqrt(aw / s[au])
    key = {(int(u), int(v)): i for i, (u, v) in enumerate(zip(au, av))}
    rev = np.array([key[(int(v), int(u))] for u, v in zip(au, av)], int)
    lam, V = np.linalg.eigh(Ad)
    psi0 = np.zeros(n, complex); psi0[wg["start"]] = np.sqrt(wg["q"])
    return dict(n=n, au=au, q=q, rev=rev, has=s > 0, lam=lam, V=V, psi0=psi0)


def _dtqw(S, psi, m):
    n, au, q = S["n"], S["au"], S["q"]; a = psi[au] * q
    for _ in range(m):
        pr = np.bincount(au, weights=(q * a).real, minlength=n) + 1j * np.bincount(au, weights=(q * a).imag, minlength=n)
        a = (2 * q * pr[au] - a)[S["rev"]]
    P = np.bincount(au, weights=np.abs(a) ** 2, minlength=n)
    pr = np.bincount(au, weights=(q * a).real, minlength=n) + 1j * np.bincount(au, weights=(q * a).imag, minlength=n)
    psi = np.where(S["has"], np.sqrt(P) * np.exp(1j * np.angle(pr)), psi)
    return psi / np.linalg.norm(psi)


def _ctqw(S, psi, gamma, t):
    psi = S["V"] @ (np.exp(-1j * gamma * S["lam"] * t) * (S["V"].conj().T @ psi))
    return psi / np.linalg.norm(psi)


def qagc_hqw_scores(S, gamma=0.05, t0=1.0, m0=1, K=5):
    """QAGC Algorithm 3 (hybrid DTQW/CTQW), our reading of the paper: K = 5 cycles, m_k = m0*k, t_k = t0*k"""
    psi = S["psi0"].copy()
    for k in range(1, K + 1):
        psi = _ctqw(S, _dtqw(S, psi, m0 * k), gamma, t0 * k)
    return np.abs(psi) ** 2


def qagc_dtqw_scores(S, m=15):
    """QAGC DTQW component alone, with the hybrid's total number of coin steps (1+2+3+4+5 = 15)"""
    return np.abs(_dtqw(S, S["psi0"].copy(), m)) ** 2


def qagc_ctqw_scores(S, gamma=0.05, t=15.0):
    """QAGC CTQW component alone (H = gamma*A, unnormalised), with the hybrid's total evolution time"""
    return np.abs(_ctqw(S, S["psi0"].copy(), gamma, t)) ** 2


def qagc_repo_scores(wg, walk_steps=100):
    """port of the AQAGC repository's qagc_baseline.py (start at node index 0, 1/sqrt(out-degree), renormalise)"""
    n = wg["n"]; succ = [set() for _ in range(n)]
    for u, v in zip(wg["src"], wg["dst"]): succ[u].add(v)
    rows, cols, vals = [], [], []
    for u in range(n):
        if not succ[u]: rows.append(u); cols.append(u); vals.append(1.0); continue
        for v in succ[u]: rows.append(v); cols.append(u); vals.append(1 / np.sqrt(len(succ[u])))
    op = sp.csr_matrix((vals, (rows, cols)), shape=(n, n)); st = np.zeros(n); st[0] = 1.0
    for _ in range(walk_steps):
        st = op @ st; nr = np.linalg.norm(st)
        if nr: st /= nr
    return st ** 2


# ---- QAGC reference-protocol classical baselines (ports of the AQAGC repository, stage06_benchmarking):
#      no attacker location; every node is a search source, as in the repository code
def _unweighted(wg):
    A_ = sp.csr_matrix((np.ones(len(wg["src"])), (wg["src"], wg["dst"])), shape=(wg["n"], wg["n"]))
    A_.sum_duplicates(); A_.data[:] = 1.0; return A_


def ref_reach_scores(wg):
    """bfs_baseline.py / astar_baseline.py: a search from EVERY node, +1 to each node it visits
    (A* there has no goal and a zero heuristic, so it visits the same nodes) = number of nodes that can reach v"""
    from scipy.sparse.csgraph import shortest_path
    D = shortest_path(_unweighted(wg), directed=True, unweighted=True)
    return np.isfinite(D).sum(axis=0).astype(float)


def ref_dfs_scores(wg, L=6):
    """dfs_baseline.py counts every simple path ending at v from every start node (exponential; it does not finish
    on our graphs). Depth-limited relaxation used here: number of directed walks of length <= L ending at v"""
    A_ = _unweighted(wg).T.tocsr(); x = np.ones(wg["n"]); tot = np.ones(wg["n"])
    for _ in range(L):
        x = A_ @ x; tot += x
    return np.log1p(tot)


def ref_markov_scores(wg, max_iter=1000, tol=1e-8):
    """markov_baseline.py: row-normalised weighted transitions, self-loop on dead ends, power iteration from uniform"""
    n = wg["n"]; W = sp.csr_matrix((wg["w"], (wg["src"], wg["dst"])), shape=(n, n)); W.sum_duplicates()
    s = np.asarray(W.sum(axis=1)).ravel(); dead = s == 0
    W = W + sp.csr_matrix((np.ones(dead.sum()), (np.flatnonzero(dead), np.flatnonzero(dead))), shape=(n, n))
    s = np.asarray(W.sum(axis=1)).ravel(); P = (sp.diags(1 / s) @ W).T.tocsr(); p = np.ones(n) / n
    for _ in range(max_iter):
        pn = P @ p
        if np.linalg.norm(pn - p) < tol: p = pn; break
        p = pn
    return p


def graded_metrics(sc, gt, seed):
    hs = sorted(gt); g = np.array([gt[h] for h in hs]); s = np.array([sc.get(h, NEG) for h in hs])
    tie = np.random.default_rng(seed).permutation(len(hs)); order = np.lexsort((tie, -s)); out = {}
    for k in (5, 10):
        kk = min(k, len(hs)); disc = 1 / np.log2(np.arange(2, kk + 2))
        out[f"NDCG@{k}"] = float((g[order[:kk]] * disc).sum() / max((np.sort(g)[::-1][:kk] * disc).sum(), 1e-12))
    kk = min(5, len(hs)); topg = set(np.lexsort((np.arange(len(hs)), -g))[:kk]); out["Top5"] = len(topg & set(order[:kk])) / kk
    fin = s > NEG / 2; sr = np.where(fin, s, (s[fin].min() - 1) if fin.any() else 0)
    rho = stats.spearmanr(sr, g).correlation; tau = stats.kendalltau(sr, g).correlation
    out["Spearman"] = 0.0 if np.isnan(rho) else float(rho); out["Kendall"] = 0.0 if np.isnan(tau) else float(tau)
    return out


SUMMARY = []


def headline(df, metric, groups, key_cols, setting):
    """QSW against the strongest member (by mean) of each baseline family; paired Wilcoxon over instances"""
    means = df.groupby("method")[metric].mean(); qm = means["QSW (w=0.5)"]
    q = df[df.method == "QSW (w=0.5)"].set_index(key_cols)[metric]; rows = []
    for gname, ms in groups.items():
        ms = [m for m in ms if m in means.index]
        if not ms: continue
        best = means[ms].idxmax(); b = df[df.method == best].set_index(key_cols)[metric].reindex(q.index)
        diff = (q - b).to_numpy(); diff = diff[~np.isnan(diff)]
        p = 1.0 if len(diff) == 0 or np.allclose(diff, 0) else stats.wilcoxon(diff, zero_method="zsplit").pvalue
        rows.append(dict(family=gname, strongest=best, family_best=means[best], QSW=qm, QSW_minus_best=qm - means[best],
                         wins=int((diff > 1e-12).sum()), losses=int((diff < -1e-12).sum()), p=p))
        SUMMARY.append(dict(setting=setting, metric=metric, family=gname, strongest=best, QSW=qm, best=means[best],
                            diff=qm - means[best], wins=int((diff > 1e-12).sum()), losses=int((diff < -1e-12).sum()), p=p))
    rank = int((means.drop([m for m in means.index if m.startswith("QW@")], errors="ignore") > qm + 1e-12).sum()) + 1
    SUMMARY.append(dict(setting=setting, metric=metric, family="(QSW rank among all methods)", strongest="", QSW=qm,
                        best=means.drop([m for m in means.index if m.startswith("QW@")], errors="ignore").max(), diff=np.nan,
                        wins=rank, losses=len([m for m in means.index if not m.startswith("QW@")]), p=np.nan))
    return pd.DataFrame(rows).set_index("family")


def subgraph(g, keep):
    keep_idx = np.flatnonzero(keep); new = -np.ones(g["n"], int); new[keep_idx] = np.arange(len(keep_idx))
    m = keep[g["par"]] & keep[g["chi"]]; g2 = dict(g)
    g2.update(n=len(keep_idx), label=[g["label"][i] for i in keep_idx], type=g["type"][keep_idx],
              par=new[g["par"][m]], chi=new[g["chi"][m]], and_p=g["and_p"][keep_idx], leaf_p=g["leaf_p"][keep_idx],
              vuln=g["vuln"][keep_idx], start=[int(new[s]) for s in g["start"]],
              host_nodes={h: [int(new[i]) for i in idx if keep[i]] for h, idx in g["host_nodes"].items()})
    g2["host_nodes"] = {h: v for h, v in g2["host_nodes"].items() if v}
    return g2


def view_noise(g, sigma, seed):
    """the defender's CVSS-derived exploit probabilities are uncertain: multiplicative log-normal noise"""
    rng = np.random.default_rng(seed); g2 = dict(g); lp = g["leaf_p"].copy()
    lp[g["vuln"]] = np.clip(lp[g["vuln"]] * np.exp(rng.normal(0, sigma, g["vuln"].sum())), 1e-4, 1.0)
    g2["leaf_p"] = lp; return g2


def view_missing(g, frac, seed):
    """the defender's scan misses a fraction of the vulnerabilities: those leaves and the steps using them vanish"""
    rng = np.random.default_rng(seed); vi = np.flatnonzero(g["vuln"])
    drop = rng.choice(vi, int(round(frac * len(vi))), replace=False) if len(vi) else np.array([], int)
    keep = np.ones(g["n"], bool); keep[drop] = False
    dset = np.zeros(g["n"], bool); dset[drop] = True
    keep[g["par"][dset[g["chi"]] & (g["type"][g["par"]] == "AND")]] = False
    return subgraph(g, keep)


Q_NAMES = {0.5: "QSW (w=0.5)", 0.0: "CTQW (w=0)", 1.0: "Markov (w=1)"}


def methods_on(gv, omegas, seed):
    wg = walk_graph(gv); T = t_rule_dir(wg)
    H, G, d = build_operators(wg["n"], wg["src"], wg["dst"], wg["w"])
    rho0 = np.zeros((wg["n"], wg["n"])); rho0[wg["start"], wg["start"]] = 1 / len(wg["start"])
    t0 = time.perf_counter(); Q = qsw_batch(H, G, d, rho0, omegas, T); tq = time.perf_counter() - t0
    sc = {f"QW@{w}": host_scores(Q[i], wg) for i, w in enumerate(omegas)}
    for w, nm in Q_NAMES.items():
        if w in omegas: sc[nm] = sc[f"QW@{w}"]
    tm = {"quantum batch": tq / len(omegas)}
    classical = {
        "BFS": lambda: host_scores(order_scores(wg, False), wg),
        "DFS": lambda: host_scores(order_scores(wg, True), wg),
        "A*": lambda: host_scores(astar_scores(wg), wg),
        "AO*": lambda: aostar_scores(gv),
        "PageRank": lambda: host_scores(pagerank_scores(wg), wg),
        "AssetRank-style": lambda: assetrank_style_scores(gv),
        **{f"MC-{S}": (lambda S=S: mc_attacker(gv, S, seed=seed * 31 + S)) for S in CFG["MC_BASE"]},
        "BFS-ref": lambda: host_scores(ref_reach_scores(wg), wg),
        "DFS-ref": lambda: host_scores(ref_dfs_scores(wg), wg),
        "A*-ref": lambda: host_scores(ref_reach_scores(wg), wg),
        "Markov-ref": lambda: host_scores(ref_markov_scores(wg), wg),
        "QAGC-repo (ref. impl.)": lambda: host_scores(qagc_repo_scores(wg), wg),
    }
    t0 = time.perf_counter(); S_ = qagc_struct(wg); t_struct = time.perf_counter() - t0
    for nm, fn in {"QAGC-DTQW": lambda: qagc_dtqw_scores(S_), "QAGC-CTQW": lambda: qagc_ctqw_scores(S_),
                   "QAGC-HQW (Alg.3)": lambda: qagc_hqw_scores(S_)}.items():
        t0 = time.perf_counter(); sc[nm] = host_scores(fn(), wg); tm[nm] = time.perf_counter() - t0 + t_struct
    for nm, fn in classical.items():
        t0 = time.perf_counter(); sc[nm] = fn(); tm[nm] = time.perf_counter() - t0
    return sc, tm, T, wg["n"], (H, G, d, rho0)


In [6]:
# 5. PART 1: MulVAL attack-graph benchmark
def bkey(b): return "Binf" if b is None else f"B{b}"


GT_KEYS = [bkey(b) for b in CFG["BUDGETS"]]
GROUPS = {"QAGC quantum walks": ["QAGC-DTQW", "QAGC-CTQW", "QAGC-HQW (Alg.3)", "QAGC-repo (ref. impl.)"],
          "QAGC reference classical baselines": ["BFS-ref", "DFS-ref", "A*-ref", "Markov-ref"],
          "Information-matched classical baselines": ["BFS", "DFS", "A*", "AO*", "PageRank", "AssetRank-style"]
                                                     + [f"MC-{S}" for S in CFG["MC_BASE"]],
          "Ablation (same model, quantum part switched)": ["CTQW (w=0)", "Markov (w=1)"]}
MAIN_METHODS = ["QSW (w=0.5)", "CTQW (w=0)", "Markov (w=1)"] + GROUPS["QAGC quantum walks"] + \
               GROUPS["QAGC reference classical baselines"] + GROUPS["Information-matched classical baselines"] + ["Random"]
P1 = None

if CFG["RUN_MULVAL"]:
    say("\n## Part 1: MulVAL multi-stage attack graphs")
    if "run_mulval_override" not in globals():
        say("[mulval]", ensure_mulval())
        XSB_BIN = os.path.dirname(glob.glob(f"{W_}/XSB/**/bin/xsb", recursive=True)[0])
        ENV = dict(os.environ, MULVALROOT=f"{W_}/mulval", PATH=f"{XSB_BIN}:{W_}/mulval/bin:{W_}/mulval/utils:" + os.environ["PATH"])
        jc = sh("readlink -f $(which javac)").stdout.strip()
        if jc: ENV["JAVA_HOME"] = os.path.dirname(os.path.dirname(jc))
        RULES = rule_probs(f"{W_}/mulval/kb/interaction_rules.P")
    else:
        run_mulval = run_mulval_override; gen_network = gen_network_override
    say(f"[plan] sizes {CFG['SIZES']} x first {CFG['N_GRAPHS']} valid seeds from {CFG['SEED0']} | ground truth "
        f"{CFG['GT_SAMPLES']} runs x attacker budgets {GT_KEYS} | QSW omega={CFG['MAIN_OMEGA']} fixed a priori")

    # ---- phase A: generate graphs (MulVAL runs in parallel on all CPU cores)
    def gen_one(H, seed):
        cf = f"{CACHE}/graph_H{H}_s{seed}.pkl"
        if os.path.exists(cf): return pickle.load(open(cf, "rb"))
        facts, meta = gen_network(H, seed)
        try:
            V, E, t = run_mulval(facts, f"{OUT}/mulval_runs/H{H}_s{seed}", CFG["MULVAL_TIMEOUT"])
            g = build_graph(V, E, meta); ok = len(g["host_nodes"]) >= 3
            rec = dict(H=H, seed=seed, ok=ok, g=g if ok else None, t_mulval=t,
                       why="" if ok else "fewer than 3 reachable hosts", steps=depth_stats(g) if ok else None)
        except Exception as e:
            rec = dict(H=H, seed=seed, ok=False, g=None, t_mulval=np.nan, why=str(e)[:160], steps=None)
        save(rec, cf); return rec

    valid = {H: [] for H in CFG["SIZES"]}; nxt = {H: CFG["SEED0"] for H in CFG["SIZES"]}; t0 = time.time()
    with ThreadPoolExecutor(max_workers=max(1, NCPU)) as ex:
        first = True
        while True:
            jobs = []
            for H in sorted(CFG["SIZES"], reverse=True):
                need = CFG["N_GRAPHS"] - len(valid[H])
                for _ in range(max(0, need + (2 if first and need > 0 else 0))):
                    if nxt[H] < CFG["SEED0"] + CFG["MAX_TRIES"]: jobs.append((H, nxt[H])); nxt[H] += 1
            first = False
            if not jobs: break
            for rec in ex.map(lambda a: gen_one(*a), jobs):
                tag = "ok" if rec["ok"] else f"skip ({rec['why']})"
                print(f"  graph H={rec['H']} seed={rec['seed']}: {tag} | MulVAL {rec['t_mulval']:.0f}s | {time.time()-t0:.0f}s")
                if rec["ok"] and len(valid[rec["H"]]) < CFG["N_GRAPHS"]: valid[rec["H"]].append(rec)
            for H in valid: valid[H].sort(key=lambda r: r["seed"])
    RECS = [r for H in CFG["SIZES"] for r in valid[H]]
    gstat = pd.DataFrame([dict(hosts=r["H"], seed=r["seed"], nodes=r["g"]["n"], AND=int((r["g"]["type"] == "AND").sum()),
                               reachable_hosts=len(r["g"]["host_nodes"]), steps_median=r["steps"][0], steps_max=r["steps"][1],
                               mulval_s=r["t_mulval"]) for r in RECS])
    table(gstat.groupby("hosts").agg(graphs=("seed", "size"), nodes=("nodes", "mean"), AND_steps=("AND", "mean"),
                                     reachable_hosts=("reachable_hosts", "mean"), steps_median=("steps_median", "median"),
                                     steps_max=("steps_max", "max"), mulval_s=("mulval_s", "mean")), "Attack graphs generated", 1)

    # ---- phase B: ground truth for every attacker model (parallel over graphs)
    def gt_one(rec):
        cf = f"{CACHE}/gt_H{rec['H']}_s{rec['seed']}.pkl"
        if os.path.exists(cf): return pickle.load(open(cf, "rb"))
        out = {bkey(b): mc_attacker(rec["g"], CFG["GT_SAMPLES"], seed=CFG["GT_SEED"] + 1000 * rec["seed"] + i, budget=b)
               for i, b in enumerate(CFG["BUDGETS"])}
        save(out, cf); return out

    t0 = time.time()
    from joblib import Parallel, delayed
    GTS = Parallel(n_jobs=max(1, NCPU), backend="loky", inner_max_num_threads=1)(delayed(gt_one)(r) for r in RECS)
    say(f"[ground truth] {len(GTS)} graphs x {len(GT_KEYS)} attacker models in {time.time()-t0:.0f}s")
    gts = pd.DataFrame([dict(hosts=r["H"], model=k, mean_p=np.mean(list(gt[k].values())), max_p=max(gt[k].values()),
                             zero_share=np.mean(np.array(list(gt[k].values())) == 0)) for r, gt in zip(RECS, GTS) for k in gt])
    table(gts.groupby(["model", "hosts"]).mean(), "Ground-truth compromise probabilities (mean / max / share of hosts never compromised)", 3)

    # ---- phase C: every method on every graph and view (cached per graph)
    def run_one(rec, gt):
        cf = f"{CACHE}/res_H{rec['H']}_s{rec['seed']}.pkl"
        if os.path.exists(cf): return pickle.load(open(cf, "rb"))
        g, seed = rec["g"], rec["seed"]
        views = {"clean": (g, CFG["OMEGAS"])}
        for s_ in CFG["NOISE"]: views[f"noise{s_}"] = (view_noise(g, s_, seed), [0.0, 0.5, 1.0])
        views[f"missing{int(CFG['MISSING'] * 100)}"] = (view_missing(g, CFG["MISSING"], seed), [0.0, 0.5, 1.0])
        rows, times, extra = [], [], {}
        for vname, (gv, oms) in views.items():
            sc, tm, T, nw, _ = methods_on(gv, oms, seed)
            if vname == "clean": extra = dict(T=T, walk_nodes=nw)
            times.append(dict(view=vname, **tm))
            for k in GT_KEYS:
                for m, s_ in sc.items():
                    rows.append(dict(H=rec["H"], seed=seed, view=vname, attacker=k, method=m, **graded_metrics(s_, gt[k], seed)))
                rng = np.random.default_rng(seed); rr = [graded_metrics({h: rng.random() for h in gt[k]}, gt[k], seed + i) for i in range(20)]
                rows.append(dict(H=rec["H"], seed=seed, view=vname, attacker=k, method="Random", **{a: np.mean([r[a] for r in rr]) for a in rr[0]}))
        out = dict(rows=rows, times=times, **extra); save(out, cf); return out

    t0 = time.time(); RES = []
    for i, (rec, gt) in enumerate(zip(RECS, GTS)):
        RES.append(run_one(rec, gt))
        r = pd.DataFrame(RES[-1]["rows"]); r = r[(r.view == "clean") & (r["attacker"] == "Binf")].set_index("method")["NDCG@10"]
        print(f"  methods H={rec['H']} seed={rec['seed']} ({i+1}/{len(RECS)}): NDCG@10 (unlimited attacker) QSW {r['QSW (w=0.5)']:.3f} "
              f"Markov {r['Markov (w=1)']:.3f} A* {r['A*']:.3f} AO* {r['AO*']:.3f} MC-100 {r['MC-100']:.3f} | {time.time()-t0:.0f}s")
    D = pd.DataFrame([x for r in RES for x in r["rows"]]); D.to_csv(f"{OUT}/part1_all_metrics.csv", index=False)
    TM = pd.DataFrame([dict(H=rec["H"], seed=rec["seed"], **t) for rec, r in zip(RECS, RES) for t in r["times"]])

    # ---- phase D: fair CPU timing + peak memory (first graph of every size), and GPU-vs-CPU agreement
    cpu_rows = []
    for H in CFG["SIZES"]:
        if not valid[H]: continue
        rec = valid[H][0]; g = rec["g"]; wg = walk_graph(g); T = t_rule_dir(wg)
        H_, G_, d_ = build_operators(wg["n"], wg["src"], wg["dst"], wg["w"])
        rho0 = np.zeros((wg["n"], wg["n"])); rho0[wg["start"], wg["start"]] = 1.0 / len(wg["start"])
        t0 = time.perf_counter(); qn = qsw_numpy(H_, G_, d_, rho0, CFG["MAIN_OMEGA"], T); tcpu = time.perf_counter() - t0
        tracemalloc.start(); qsw_numpy(H_, G_, d_, rho0, CFG["MAIN_OMEGA"], T); mq = tracemalloc.get_traced_memory()[1] / 2**20; tracemalloc.stop()
        qb = qsw_batch(H_, G_, d_, rho0, [CFG["MAIN_OMEGA"]], T)[0]
        row = dict(hosts=H, walk_nodes=wg["n"], **{"QSW cpu_s": tcpu, "QSW peak_MB": mq, "GPU-vs-CPU max diff": np.abs(qb - qn).max()})
        for nm, fn in {"A*": lambda: astar_scores(wg), "AO*": lambda: aostar_scores(g), "PageRank": lambda: pagerank_scores(wg),
                       "MC-1000": lambda: mc_attacker(g, 1000, seed=1)}.items():
            tracemalloc.start(); t0 = time.perf_counter(); fn(); el = time.perf_counter() - t0
            row[f"{nm} cpu_s"] = el; row[f"{nm} peak_MB"] = tracemalloc.get_traced_memory()[1] / 2**20; tracemalloc.stop()
        cpu_rows.append(row)
    CPU1 = pd.DataFrame(cpu_rows).set_index("hosts")

    # ---- phase E: report
    pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)
    Mx = ["NDCG@5", "NDCG@10", "Top5", "Spearman", "Kendall"]

    def stats_vs(df, ref="QSW (w=0.5)", against=None, metrics_=("NDCG@10", "Spearman")):
        q = df[df.method == ref].set_index(["H", "seed"]); out = []
        for b in (against or [m for m in MAIN_METHODS if m != ref]):
            bb = df[df.method == b].set_index(["H", "seed"]).reindex(q.index)
            for mt in metrics_:
                diff = (q[mt] - bb[mt]).to_numpy(); diff = diff[~np.isnan(diff)]
                p = 1.0 if len(diff) == 0 or np.allclose(diff, 0) else stats.wilcoxon(diff, zero_method="zsplit").pvalue
                bs = np.random.default_rng(1).choice(diff, (2000, len(diff))).mean(1) if len(diff) else np.zeros(1)
                out.append(dict(vs=b, metric=mt, mean_diff=diff.mean() if len(diff) else 0, ci_lo=np.percentile(bs, 2.5),
                                ci_hi=np.percentile(bs, 97.5), wins=int((diff > 1e-12).sum()), ties=int((abs(diff) <= 1e-12).sum()),
                                losses=int((diff < -1e-12).sum()), p=p))
        out = pd.DataFrame(out)
        for mt in metrics_:
            k_ = out.metric == mt; ps = out.loc[k_, "p"].to_numpy(); o = np.argsort(ps); adj = np.empty(len(ps)); run = 0
            for r_, i in enumerate(o): run = max(run, (len(ps) - r_) * ps[i]); adj[i] = min(1, run)
            out.loc[k_, "p_holm"] = adj
        return out

    for k in GT_KEYS:
        c = D[(D.view == "clean") & (D["attacker"] == k)]
        table(c.groupby("method")[Mx].mean().reindex(MAIN_METHODS), f"[{k}] clean graphs: mean over all {c.seed.nunique()} graphs")
        table(c.pivot_table(index="method", columns="H", values="NDCG@10").reindex(MAIN_METHODS), f"[{k}] NDCG@10 by network size (hosts)")
        table(headline(c, "NDCG@10", GROUPS, ["H", "seed"], f"MulVAL {k} clean"), f"[{k}] HEADLINE: QSW vs the strongest method of each family (NDCG@10)")
        table(headline(c, "Spearman", GROUPS, ["H", "seed"], f"MulVAL {k} clean"), f"[{k}] HEADLINE: QSW vs the strongest method of each family (Spearman)")
        st = stats_vs(c); st.to_csv(f"{OUT}/part1_stats_{k}.csv", index=False)
        table(st.set_index(["vs", "metric"]), f"[{k}] QSW minus each method (paired over graphs, Wilcoxon + Holm, bootstrap 95% CI)")
    # quantum contribution: does coherence change the ranking quality, all attacker models and views?
    qc = []
    for k in GT_KEYS:
        for v in D.view.unique():
            s_ = stats_vs(D[(D.view == v) & (D["attacker"] == k)], ref="QSW (w=0.5)", against=["Markov (w=1)", "CTQW (w=0)"], metrics_=("NDCG@10",))
            s2 = stats_vs(D[(D.view == v) & (D["attacker"] == k)], ref="CTQW (w=0)", against=["Markov (w=1)"], metrics_=("NDCG@10",))
            s_["comparison"] = "QSW - " + s_["vs"]; s2["comparison"] = "CTQW - Markov"
            for x in (s_, s2):
                x["attacker"] = k; x["view"] = v; qc.append(x)
    QC = pd.concat(qc); QC.to_csv(f"{OUT}/part1_quantum_contribution.csv", index=False)
    table(QC.set_index(["attacker", "view", "comparison"])[["mean_diff", "ci_lo", "ci_hi", "wins", "losses", "p"]],
          "Quantum contribution: QSW / CTQW minus the classical walk (Markov, omega=1) and QSW minus CTQW, NDCG@10")
    OM = D[(D.view == "clean") & (D.method.str.startswith("QW@"))].copy(); OM["omega"] = OM.method.str[3:].astype(float)
    table(OM.pivot_table(index="omega", columns="attacker", values="NDCG@10").T, "Decoherence curve (clean, mean NDCG@10; omega=0 pure quantum ... 1 classical)")
    table(OM.pivot_table(index="omega", columns="attacker", values="Spearman").T, "Decoherence curve (clean, mean Spearman)")
    rob = D[D.method.isin(MAIN_METHODS)].pivot_table(index=["attacker", "method"], columns="view", values="NDCG@10")
    table(rob.reindex(pd.MultiIndex.from_product([GT_KEYS, MAIN_METHODS])), "Robustness: mean NDCG@10 per defender view (noise = CVSS uncertainty, missing = 20% of vulnerabilities unseen)")
    for k in GT_KEYS:
        for v in [x for x in D.view.unique() if x != "clean"]:
            headline(D[(D.view == v) & (D["attacker"] == k)], "NDCG@10", GROUPS, ["H", "seed"], f"MulVAL {k} {v}")
    rank_rows = []
    for (k, v), grp in D[D.method.isin(MAIN_METHODS)].groupby(["attacker", "view"]):
        mm = grp.groupby("method")["NDCG@10"].mean().sort_values(ascending=False)
        rank_rows.append(dict(attacker=k, view=v, QSW_rank=int(list(mm.index).index("QSW (w=0.5)") + 1), of=len(mm), best=mm.index[0],
                              best_NDCG10=mm.iloc[0], QSW_NDCG10=mm["QSW (w=0.5)"]))
    table(pd.DataFrame(rank_rows).set_index(["attacker", "view"]), "Where QSW ranks among all methods (mean NDCG@10)")
    table(TM.groupby(["H"]).mean(numeric_only=True).drop(columns=["seed"], errors="ignore").T, "Run time per graph in seconds (quantum batch = GPU time per omega)")
    table(CPU1.T, "Fair CPU timing, peak memory (first graph of each size) and GPU-vs-CPU agreement")
    P1 = dict(D=D, TM=TM, CPU=CPU1, QC=QC, OM=OM, gstat=gstat)



## Part 1: MulVAL multi-stage attack graphs
[mulval] built
[plan] sizes [25, 50, 75, 100] x first 10 valid seeds from 200 | ground truth 20000 runs x attacker budgets ['B10', 'B30', 'Binf'] | QSW omega=0.5 fixed a priori
  graph H=100 seed=200: ok | MulVAL 270s | 270s
  graph H=100 seed=201: ok | MulVAL 1121s | 1121s
  graph H=100 seed=202: ok | MulVAL 133s | 1121s
  graph H=100 seed=203: ok | MulVAL 662s | 1121s
  graph H=100 seed=204: ok | MulVAL 1091s | 1225s
  graph H=100 seed=205: ok | MulVAL 905s | 1225s
  graph H=100 seed=206: ok | MulVAL 538s | 1225s
  graph H=100 seed=207: ok | MulVAL 91s | 1225s
  graph H=100 seed=208: ok | MulVAL 336s | 1511s
  graph H=100 seed=209: ok | MulVAL 376s | 1576s
  graph H=100 seed=210: ok | MulVAL 347s | 1576s
  graph H=100 seed=211: ok | MulVAL 178s | 1576s
  graph H=75 seed=200: ok | MulVAL 13s | 1576s
  graph H=75 seed=201: ok | MulVAL 908s | 2324s
  graph H=75 seed=202: ok | MulVAL 45s | 2324s
  graph H=75 seed=203: ok | MulVAL 133s | 2324s


In [7]:
# 6. PART 2: flow datasets (CIC-IDS2017 / CSE-CIC-IDS2018), strictly unsupervised protocol v4.1
COLS = ["Src IP", "Dst IP", "Timestamp", "Label", "Attempted Category"]
FLOW_SPEC = {"CIC2017": dict(subdir="CICIDS2017_improved", internal=["192.168.10.0/24"],
                             files=["monday", "tuesday", "wednesday", "thursday", "friday"]),
             "CSE2018": dict(subdir="CSECICIDS2018_improved", internal=["172.31.0.0/16"], files=None)}
KS = (1, 3, 5, 10); FLOW_M = [f"R@{k}" for k in KS] + ["MRR", "MAP", "NDCG@10"]
FLOW_METHODS = ["QSW (w=0.5)", "CTQW (w=0)", "Markov (w=1)", "QAGC-DTQW", "QAGC-CTQW", "QAGC-HQW (Alg.3)",
                "QAGC-repo (ref. impl.)", "BFS-ref", "DFS-ref", "A*-ref", "Markov-ref", "BFS", "DFS", "A*", "PageRank",
                "WDegree", "Random"]
FLOW_GROUPS = {"QAGC quantum walks": ["QAGC-DTQW", "QAGC-CTQW", "QAGC-HQW (Alg.3)", "QAGC-repo (ref. impl.)"],
               "QAGC reference classical baselines": ["BFS-ref", "DFS-ref", "A*-ref", "Markov-ref"],
               "Information-matched classical baselines": ["BFS", "DFS", "A*", "PageRank", "WDegree"],
               "Ablation (same model, quantum part switched)": ["CTQW (w=0)", "Markov (w=1)"]}


def flow_load_day(path):
    try:
        df = pd.read_csv(path, usecols=COLS, engine="pyarrow")
    except Exception:
        df = pd.read_csv(path, usecols=COLS, low_memory=False)
    df["Timestamp"] = pd.to_datetime(df["Timestamp"], format="ISO8601", errors="coerce")
    df = df.dropna(subset=["Timestamp", "Src IP", "Dst IP"])
    df["Src IP"] = df["Src IP"].astype(str).str.strip(); df["Dst IP"] = df["Dst IP"].astype(str).str.strip()
    lab = df["Label"].astype(str).str.strip().str.upper(); df["mal"] = lab.ne("BENIGN")
    df["succ"] = df["mal"] & (pd.to_numeric(df["Attempted Category"], errors="coerce") == -1)
    return df[["Src IP", "Dst IP", "Timestamp", "mal", "succ"]]


def flow_internal_map(ips, nets):
    nets = [ipaddress.ip_network(x) for x in nets]; out = {}
    for ip in ips:
        try: a = ipaddress.ip_address(ip); out[ip] = any(a in nt for nt in nets)
        except ValueError: out[ip] = False
    return out


def flow_instance(w, N, is_int, ext_mal, meta):
    """label-free graph: ALL active internal hosts, then external initiators (entries), then busiest others, up to N"""
    act = pd.concat([w["Src IP"], w["Dst IP"]]).value_counts()
    if len(act) < N: return None, "too_small"
    act = act.rename_axis("ip").reset_index(name="c"); act["internal"] = act["ip"].map(is_int).fillna(False).astype(bool)
    n_int = int(act["internal"].sum())
    if n_int >= N: return None, "internal_hosts_exceed_N"
    si = w["Src IP"].map(is_int).fillna(False).astype(bool); di = w["Dst IP"].map(is_int).fillna(False).astype(bool)
    initiators = set(w.loc[~si & di, "Src IP"])
    act["prio"] = np.where(act["internal"], 2, np.where(act["ip"].isin(initiators), 1, 0))
    act = act.sort_values(["prio", "c", "ip"], ascending=[False, False, True]).head(N)
    names = act["ip"].to_numpy(); pos = {ip: i for i, ip in enumerate(names)}
    sel = w[w["Src IP"].isin(pos) & w["Dst IP"].isin(pos) & (w["Src IP"] != w["Dst IP"])]
    e = sel.groupby(["Src IP", "Dst IP"]).size().reset_index(name="c")
    if len(e) == 0: return None, "no_edges"
    src = e["Src IP"].map(pos).to_numpy(int); dst = e["Dst IP"].map(pos).to_numpy(int); wgt = np.log1p(e["c"].to_numpy(float))
    internal = np.array([bool(is_int.get(ip, False)) for ip in names])
    entries = sorted({int(u) for u, v in zip(src, dst) if not internal[u] and internal[v]})
    if not entries: return None, "no_entry"
    succ = w[w["succ"]]
    gt_ips = {ip for ip in pd.unique(pd.concat([succ["Src IP"], succ["Dst IP"]])) if is_int.get(ip, False)}
    cand = internal.copy(); gt = np.array([ip in gt_ips for ip in names]) & cand
    if gt.sum() == 0: return None, "gt_not_in_graph"
    if cand.sum() < 2: return None, "few_candidates"
    s_out = np.bincount(src, weights=wgt, minlength=N)[entries]; q = s_out / s_out.sum()
    seed = zlib.crc32(f"{meta['dataset']}|{meta['day']}|{meta['window']}|{N}".encode())
    return dict(meta, N=N, n=N, names=names, src=src, dst=dst, w=wgt, start=entries, q=q, internal=internal, cand=cand, gt=gt,
                seed=seed, tieperm=np.random.default_rng(seed).permutation(N), n_edges=len(e), n_entries=len(entries),
                n_gt=int(gt.sum()), n_cand=int(cand.sum())), "ok"


def flow_t_rule(inst):
    adj = [[] for _ in range(inst["n"])]
    for a, b in zip(inst["src"], inst["dst"]): adj[a].append(b); adj[b].append(a)
    dist = np.full(inst["n"], -1); q = deque(inst["start"])
    for s in inst["start"]: dist[s] = 0
    while q:
        u = q.popleft()
        for v in adj[u]:
            if dist[v] < 0: dist[v] = dist[u] + 1; q.append(v)
    r = dist[dist > 0]
    return max(DT, round(max(float(r.mean()) if len(r) else 1.0, 1.0) / DT) * DT)


def rank_metrics(score, cand, gt, tieperm):
    s = np.where(np.isfinite(score), score, NEG); idx = np.flatnonzero(cand)
    order = idx[np.lexsort((tieperm[idx], -s[idx]))]; hit = gt[order]; ng = int(gt.sum()); ranks = np.flatnonzero(hit) + 1
    out = {f"R@{k}": hit[:k].sum() / ng for k in KS}
    out["MRR"] = 1.0 / ranks[0] if len(ranks) else 0.0
    out["MAP"] = float(np.mean(np.arange(1, len(ranks) + 1) / ranks)) if len(ranks) else 0.0
    out["NDCG@10"] = np.sum(1 / np.log2(ranks[ranks <= 10] + 1)) / np.sum(1 / np.log2(np.arange(1, min(ng, 10) + 1) + 1))
    return out


def flow_dataset(name):
    spec = FLOW_SPEC[name]; base = os.path.join(CFG["FLOW_ROOT"], spec["subdir"])
    files = spec["files"] or sorted(f[:-4] for f in os.listdir(base) if f.endswith(".csv"))
    insts, log = [], []
    for day in files:
        t0 = time.time(); df = flow_load_day(os.path.join(base, f"{day}.csv"))
        ips = pd.unique(pd.concat([df["Src IP"], df["Dst IP"]])); is_int = flow_internal_map(ips, spec["internal"])
        mal = df[df["mal"]]; ext_mal = {ip for ip in pd.unique(pd.concat([mal["Src IP"], mal["Dst IP"]])) if not is_int[ip]}
        nd = 0
        for N, freq in CFG["FLOW_SIZES"].items():
            df["win"] = df["Timestamp"].dt.floor(freq); att = np.sort(df.loc[df["succ"], "win"].unique())
            sub = df[df["win"].isin(att)]; ok, why = [], {}
            for wk, w in sub.groupby("win", sort=True):
                inst, r = flow_instance(w, N, is_int, ext_mal, dict(dataset=name, day=day, window=str(wk)))
                why[r] = why.get(r, 0) + 1
                if inst is not None: ok.append(inst)
            if len(ok) > CFG["FLOW_MAXWIN"]:
                ok = [ok[i] for i in np.unique(np.linspace(0, len(ok) - 1, CFG["FLOW_MAXWIN"]).round().astype(int))]
            insts += ok; nd += len(ok)
            log.append(dict(dataset=name, day=day, N=N, attack_windows=len(att), used=len(ok), **{f"skip_{k}": v for k, v in why.items() if k != "ok"}))
        print(f"  [{name}/{day}] rows={len(df):,} instances={nd} ({time.time()-t0:.0f}s)")
        del df, sub; gc.collect()
    for i in insts: i["uid"] = f"{name}|{i['day']}|{i['window']}|{i['N']}"
    return insts, pd.DataFrame(log).fillna(0)


def flow_methods(inst):
    wg = dict(n=inst["n"], src=inst["src"], dst=inst["dst"], w=inst["w"], start=inst["start"], q=inst["q"])
    H, G, d = build_operators(inst["n"], inst["src"], inst["dst"], inst["w"]); T = flow_t_rule(inst)
    rho0 = np.zeros((inst["n"], inst["n"])); rho0[inst["start"], inst["start"]] = inst["q"]
    t0 = time.perf_counter(); Q = qsw_batch(H, G, d, rho0, CFG["OMEGAS"], T); tq = (time.perf_counter() - t0) / len(CFG["OMEGAS"])
    sc = {f"QW@{w}": Q[i] for i, w in enumerate(CFG["OMEGAS"])}
    for w, nm in Q_NAMES.items(): sc[nm] = sc[f"QW@{w}"]
    tm = {"quantum batch": tq}
    s_in = np.bincount(inst["dst"], weights=inst["w"], minlength=inst["n"]); s_out = np.bincount(inst["src"], weights=inst["w"], minlength=inst["n"])
    for nm, fn in {"BFS": lambda: order_scores(wg, False, undirected=True), "DFS": lambda: order_scores(wg, True, undirected=True),
                   "A*": lambda: astar_scores(wg, undirected=True, normalise=True), "PageRank": lambda: pagerank_scores(wg),
                   "WDegree": lambda: s_in + s_out, "QAGC-repo (ref. impl.)": lambda: qagc_repo_scores(wg),
                   "BFS-ref": lambda: ref_reach_scores(wg), "DFS-ref": lambda: ref_dfs_scores(wg),
                   "A*-ref": lambda: ref_reach_scores(wg), "Markov-ref": lambda: ref_markov_scores(wg)}.items():
        t0 = time.perf_counter(); sc[nm] = fn(); tm[nm] = time.perf_counter() - t0
    t0 = time.perf_counter(); S_ = qagc_struct(wg); t_struct = time.perf_counter() - t0
    for nm, fn in {"QAGC-DTQW": lambda: qagc_dtqw_scores(S_), "QAGC-CTQW": lambda: qagc_ctqw_scores(S_),
                   "QAGC-HQW (Alg.3)": lambda: qagc_hqw_scores(S_)}.items():
        t0 = time.perf_counter(); sc[nm] = fn(); tm[nm] = time.perf_counter() - t0 + t_struct
    return sc, tm, T


P2 = {}
if CFG["RUN_FLOW"]:
    say("\n## Part 2: flow datasets (strictly unsupervised protocol v4.1)")
    for name in CFG["FLOW_DATASETS"]:
        if not os.path.isdir(os.path.join(CFG["FLOW_ROOT"], FLOW_SPEC[name]["subdir"])):
            say(f"[flow] {name}: data folder not found, skipped"); continue
        cf = f"{CACHE}/flow_{name}.pkl"
        if os.path.exists(cf):
            F = pickle.load(open(cf, "rb"))
        else:
            insts, log = flow_dataset(name); rows, times = [], []
            t0 = time.time()
            for j, inst in enumerate(insts):
                sc, tm, T = flow_methods(inst); times.append(dict(N=inst["N"], **tm))
                prng = np.random.default_rng(inst["seed"] + 99); gp = np.zeros(inst["n"], bool)
                gp[prng.choice(np.flatnonzero(inst["cand"]), inst["n_gt"], replace=False)] = True
                for m, s_ in sc.items():
                    r = rank_metrics(s_, inst["cand"], inst["gt"], inst["tieperm"]); rp = rank_metrics(s_, inst["cand"], gp, inst["tieperm"])
                    rows.append(dict(uid=inst["uid"], day=inst["day"], N=inst["N"], method=m, perm_MAP=rp["MAP"], **r))
                rr = [rank_metrics(np.random.default_rng(inst["seed"] + i).random(inst["n"]), inst["cand"], inst["gt"], inst["tieperm"]) for i in range(30)]
                rows.append(dict(uid=inst["uid"], day=inst["day"], N=inst["N"], method="Random", perm_MAP=np.mean([x["MAP"] for x in rr]),
                                 **{k: np.mean([x[k] for x in rr]) for k in rr[0]}))
                if (j + 1) % 50 == 0: print(f"  [{name}] {j+1}/{len(insts)} instances ({time.time()-t0:.0f}s)")
            F = dict(D=pd.DataFrame(rows), TM=pd.DataFrame(times), log=log,
                     meta=pd.DataFrame([{k: i[k] for k in ("uid", "day", "N", "n_edges", "n_entries", "n_gt", "n_cand")} for i in insts]))
            save(F, cf)
        F["D"].to_csv(f"{OUT}/part2_{name}_metrics.csv", index=False); P2[name] = F
        Df = F["D"]
        table(F["log"].set_index(["day", "N"]).drop(columns=["dataset"]), f"{name}: attack windows -> usable instances (skip reasons)", 0)
        table(F["meta"].groupby("N")[["n_cand", "n_gt", "n_entries"]].mean(), f"{name}: mean internal candidates / attacked hosts / entry nodes per graph", 1)
        for N in sorted(Df.N.unique()):
            t = Df[Df.N == N]
            tab = t.groupby("method")[FLOW_M].mean().reindex(FLOW_METHODS)
            tab["time_ms"] = F["TM"][F["TM"].N == N].mean(numeric_only=True).reindex(["quantum batch" if m in Q_NAMES.values() else m for m in FLOW_METHODS]).to_numpy() * 1e3
            table(tab, f"{name} N={N}: mean over {t.uid.nunique()} windows (time_ms: quantum = GPU per omega)")
            table(t.pivot_table(index="method", columns="day", values="MAP").reindex(FLOW_METHODS), f"{name} N={N}: MAP per day", 3)
            table(t.groupby("method")[["perm_MAP"]].mean().reindex(FLOW_METHODS).T, f"{name} N={N}: label-permutation control (all should be near Random)", 3)
            table(headline(t, "MAP", FLOW_GROUPS, ["uid"], f"{name} N={N}"), f"{name} N={N} HEADLINE: QSW vs the strongest method of each family (MAP)")
            q = t[t.method == "QSW (w=0.5)"].set_index("uid"); st = []
            for b in FLOW_METHODS[1:]:
                bb = t[t.method == b].set_index("uid").reindex(q.index)
                for mt in ("MAP", "MRR"):
                    diff = (q[mt] - bb[mt]).to_numpy()
                    p = 1.0 if np.allclose(diff, 0) else stats.wilcoxon(diff, zero_method="zsplit").pvalue
                    bs = np.random.default_rng(1).choice(diff, (2000, len(diff))).mean(1)
                    st.append(dict(vs=b, metric=mt, mean_diff=diff.mean(), ci_lo=np.percentile(bs, 2.5), ci_hi=np.percentile(bs, 97.5),
                                   wins=int((diff > 1e-12).sum()), losses=int((diff < -1e-12).sum()), p=p))
            st = pd.DataFrame(st)
            for mt in ("MAP", "MRR"):
                k_ = st.metric == mt; ps = st.loc[k_, "p"].to_numpy(); o = np.argsort(ps); adj = np.empty(len(ps)); run = 0
                for r_, i in enumerate(o): run = max(run, (len(ps) - r_) * ps[i]); adj[i] = min(1, run)
                st.loc[k_, "p_holm"] = adj
            table(st.set_index(["vs", "metric"]), f"{name} N={N}: QSW minus each method (Wilcoxon + Holm, bootstrap 95% CI)")
        om = Df[Df.method.str.startswith("QW@")].copy(); om["omega"] = om.method.str[3:].astype(float)
        table(om.pivot_table(index="N", columns="omega", values="MAP"), f"{name}: decoherence curve (mean MAP; omega=0 quantum ... 1 classical)", 3)



## Part 2: flow datasets (strictly unsupervised protocol v4.1)
  [CIC2017/monday] rows=371,624 instances=0 (3s)
  [CIC2017/tuesday] rows=322,078 instances=51 (4s)
  [CIC2017/wednesday] rows=496,641 instances=50 (6s)
  [CIC2017/thursday] rows=362,076 instances=45 (5s)
  [CIC2017/friday] rows=547,557 instances=50 (7s)
  [CIC2017] 50/196 instances (20s)
  [CIC2017] 100/196 instances (44s)
  [CIC2017] 150/196 instances (66s)

### CIC2017: attack windows -> usable instances (skip reasons)
```
               attack_windows  used  skip_too_small  skip_no_entry
day       N                                                       
monday    50                0     0             0.0            0.0
          200               0     0             0.0            0.0
          500               0     0             0.0            0.0
tuesday   50               29    20             0.0            0.0
          200              29    20             1.0            0.0
          500              11    11  

In [8]:
# 7. figures (paper-ready PNG + PDF)
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 9, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.color": "#e6e6e3", "grid.linewidth": 0.6, "axes.edgecolor": "#8a8a85", "legend.frameon": False,
                     "figure.dpi": 150, "savefig.bbox": "tight"})
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]   # fixed categorical order
STYLE = {"QSW (w=0.5)": (SERIES[0], "o"), "Markov (w=1)": (SERIES[1], "s"), "CTQW (w=0)": (SERIES[2], "^"),
         "A*": (SERIES[3], "D"), "AO*": (SERIES[4], "v"), "PageRank": (SERIES[5], "P"), "MC-100": (SERIES[6], "X"),
         "QAGC-HQW (Alg.3)": (SERIES[7], "*")}


def savefig(fig, name):
    fig.savefig(f"{FIG}/{name}.png"); fig.savefig(f"{FIG}/{name}.pdf"); plt.close(fig); say(f"[figure] {FIG}/{name}.png")


if P1 is not None:
    OMt = P1["OM"]
    fig, ax = plt.subplots(figsize=(4.2, 2.8))
    for i, k in enumerate(GT_KEYS):
        y = OMt[OMt["attacker"] == k].groupby("omega")["NDCG@10"].mean()
        ax.plot(y.index, y.values, marker="os^"[i % 3], color=SERIES[i], lw=2, ms=4, label={"Binf": "unlimited attacker"}.get(k, f"attacker budget {k[1:]}"))
    ax.set_xlabel("decoherence rate omega (0 = pure quantum, 1 = classical walk)"); ax.set_ylabel("mean NDCG@10")
    ax.legend(fontsize=8); ax.set_title("Ranking quality across the quantum-classical range", fontsize=9, loc="left")
    savefig(fig, "fig1_decoherence_mulval")

    Dt = P1["D"]; key = [m for m in STYLE if m in Dt.method.unique()]
    views = list(Dt.view.unique())
    fig, axes = plt.subplots(1, len(GT_KEYS), figsize=(3.0 * len(GT_KEYS), 2.9), sharey=True)
    for a, k in zip(np.atleast_1d(axes), GT_KEYS):
        for m in key:
            y = Dt[(Dt["attacker"] == k) & (Dt.method == m)].groupby("view")["NDCG@10"].mean().reindex(views)
            a.plot(range(len(views)), y.values, marker=STYLE[m][1], color=STYLE[m][0], lw=2, ms=4, label=m)
        a.set_xticks(range(len(views))); a.set_xticklabels([{"clean": "clean"}.get(v, v.replace("noise", "CVSS noise ") if v.startswith("noise") else v.replace("missing", "") + "% vulns unseen") for v in views], rotation=30, ha="right", fontsize=7)
        a.set_title({"Binf": "unlimited attacker"}.get(k, f"budget {k[1:]}"), fontsize=9, loc="left")
    np.atleast_1d(axes)[0].set_ylabel("mean NDCG@10")
    h_, l_ = np.atleast_1d(axes)[0].get_legend_handles_labels(); fig.legend(h_, l_, loc="center left", ncol=1, fontsize=7, bbox_to_anchor=(1.0, 0.5))
    savefig(fig, "fig2_robustness_mulval")

    fig, axes = plt.subplots(1, len(GT_KEYS), figsize=(3.0 * len(GT_KEYS), 2.8), sharey=True)
    for a, k in zip(np.atleast_1d(axes), GT_KEYS):
        for m in key:
            y = Dt[(Dt["attacker"] == k) & (Dt.view == "clean") & (Dt.method == m)].groupby("H")["NDCG@10"].mean()
            a.plot(y.index, y.values, marker=STYLE[m][1], color=STYLE[m][0], lw=2, ms=4, label=m)
        a.set_xlabel("network size (hosts)"); a.set_title({"Binf": "unlimited attacker"}.get(k, f"budget {k[1:]}"), fontsize=9, loc="left")
    np.atleast_1d(axes)[0].set_ylabel("mean NDCG@10")
    h_, l_ = np.atleast_1d(axes)[0].get_legend_handles_labels(); fig.legend(h_, l_, loc="center left", ncol=1, fontsize=7, bbox_to_anchor=(1.0, 0.5))
    savefig(fig, "fig3_size_mulval")

    C1 = P1["CPU"]
    fig, ax = plt.subplots(figsize=(4.2, 2.8))
    for i, m in enumerate(["QSW", "A*", "AO*", "PageRank", "MC-1000"]):
        col = f"{m} cpu_s"
        if col in C1: ax.plot(C1["walk_nodes"], C1[col], marker="osDPX"[i], color=SERIES[[0, 3, 4, 5, 6][i]], lw=2, ms=4, label=m)
    ax.set_yscale("log"); ax.set_xlabel("attack-graph nodes (walk graph)"); ax.set_ylabel("CPU seconds per graph (log)")
    ax.legend(fontsize=7); ax.set_title("Cost on classical hardware", fontsize=9, loc="left")
    savefig(fig, "fig4_cpu_time")

if P2:
    fig, axes = plt.subplots(1, len(P2), figsize=(3.4 * len(P2), 2.8), sharey=True)
    for a, (name, F) in zip(np.atleast_1d(axes), P2.items()):
        om = F["D"][F["D"].method.str.startswith("QW@")].copy(); om["omega"] = om.method.str[3:].astype(float)
        for i, N in enumerate(sorted(om.N.unique())):
            y = om[om.N == N].groupby("omega")["MAP"].mean()
            a.plot(y.index, y.values, marker="os^"[i % 3], color=SERIES[i], lw=2, ms=4, label=f"N = {N}")
        a.set_xlabel("decoherence rate omega"); a.set_title(name, fontsize=9, loc="left")
    np.atleast_1d(axes)[0].set_ylabel("mean MAP"); np.atleast_1d(axes)[-1].legend(fontsize=7)
    savefig(fig, "fig5_decoherence_flow")

def bars(means, title, xlabel, name):
    means = means.drop([m for m in means.index if m.startswith("QW@")], errors="ignore").sort_values()
    fig, ax = plt.subplots(figsize=(4.4, 0.22 * len(means) + 0.8))
    cols = [SERIES[0] if m == "QSW (w=0.5)" else "#b4b4ad" for m in means.index]
    ax.barh(range(len(means)), means.values, color=cols, height=0.7)
    ax.set_yticks(range(len(means))); ax.set_yticklabels(means.index, fontsize=7); ax.set_xlabel(xlabel)
    ax.grid(axis="y", visible=False); ax.set_title(title, fontsize=9, loc="left")
    for i, v in enumerate(means.values):
        if means.index[i] == "QSW (w=0.5)": ax.text(v, i, f" {v:.3f}", va="center", fontsize=7)
    savefig(fig, name)


if P1 is not None:
    for k in GT_KEYS:
        c = P1["D"][(P1["D"].view == "clean") & (P1["D"]["attacker"] == k)]
        bars(c.groupby("method")["NDCG@10"].mean(), {"Binf": "MulVAL graphs, unlimited attacker"}.get(k, f"MulVAL graphs, attacker budget {k[1:]}"),
             "mean NDCG@10", f"fig6_methods_mulval_{k}")
for name, F in P2.items():
    for N in sorted(F["D"].N.unique()):
        t = F["D"][F["D"].N == N]; bars(t.groupby("method")["MAP"].mean(), f"{name}, N = {N}", "mean MAP", f"fig7_methods_{name}_N{N}")

if SUMMARY:
    SM = pd.DataFrame(SUMMARY); SM.to_csv(f"{OUT}/PAPER_SUMMARY.csv", index=False)
    say("\n## PAPER SUMMARY (QSW minus the strongest method of each family; positive = QSW better)")
    piv = SM[SM.family != "(QSW rank among all methods)"].pivot_table(index=["setting", "metric"], columns="family", values="diff", aggfunc="first")
    table(piv, "QSW minus strongest of each family, every setting", 3)
    rk = SM[SM.family == "(QSW rank among all methods)"].set_index(["setting", "metric"])[["QSW", "best", "wins", "losses"]]
    rk.columns = ["QSW", "best_any_method", "QSW_rank", "of_methods"]; table(rk, "QSW rank among all methods, every setting", 3)
say(f"\nAll outputs in {OUT}: RESULTS.md, PAPER_SUMMARY.csv, part1_*.csv, part2_*.csv, figures/*.png|pdf")


[figure] /kaggle/working/qsaw_final/figures/fig1_decoherence_mulval.png
[figure] /kaggle/working/qsaw_final/figures/fig2_robustness_mulval.png
[figure] /kaggle/working/qsaw_final/figures/fig3_size_mulval.png
[figure] /kaggle/working/qsaw_final/figures/fig4_cpu_time.png
[figure] /kaggle/working/qsaw_final/figures/fig5_decoherence_flow.png
[figure] /kaggle/working/qsaw_final/figures/fig6_methods_mulval_B10.png
[figure] /kaggle/working/qsaw_final/figures/fig6_methods_mulval_B30.png
[figure] /kaggle/working/qsaw_final/figures/fig6_methods_mulval_Binf.png
[figure] /kaggle/working/qsaw_final/figures/fig7_methods_CIC2017_N50.png
[figure] /kaggle/working/qsaw_final/figures/fig7_methods_CIC2017_N200.png
[figure] /kaggle/working/qsaw_final/figures/fig7_methods_CIC2017_N500.png
[figure] /kaggle/working/qsaw_final/figures/fig7_methods_CSE2018_N500.png

## PAPER SUMMARY (QSW minus the strongest method of each family; positive = QSW better)

### QSW minus strongest of each family, every setting
```
